# SiteWise AI: EuroSAT Land Classification with PAPNet (Pattern-Affinitive Propagation Network)

This notebook implements the complete **PAPNet (Pattern-Affinitive Propagation Network)** pipeline for EuroSAT land use classification, downstream site suitability (Model 2), and multi-criteria site recommendation (Model 3).

### What is PAPNet?
Unlike standard convolutional networks like EfficientNet or ResNet that rely exclusively on local spatial convolutions, **PAPNet** models **non-local pattern affinities** across satellite imagery:
- Satellite scenes (canopy textures, agricultural crop geometries, urban structures, river curves, road networks) have rich structural correlations across disjoint spatial regions.
- **Pattern Affinity Matrix**: Computes pairwise affinity across all spatial patches:
  $$A = \text{Softmax}\left(\frac{Q \cdot K^T}{\sqrt{d}}\right)$$
- **Pattern-Affinitive Propagation**: Propagates context-rich pattern representations across the feature map:
  $$P = A \cdot V$$
  $$Y = X + \gamma \cdot W_o(P)$$
  where $\gamma$ is a learnable gating parameter initialized to $0$ for smooth convergence.

---
### End-to-End Pipeline:
1. **Model 1 (PAPNet)**: Satellite Image $(64\times 64\times 3) \to$ 10 Land Class Probabilities
2. **Visual Feature Extraction**: Computes 8 physical indices (Mean RGB, Brightness, Vegetation, Water, Built-up, Road Indicator)
3. **Model 2 (Site Suitability)**: 18 features (8 visual + 10 PAPNet) $\to$ Suitability Score for target facility (Bank, Hospital, etc.)
4. **Model 3 (Site Recommender)**: 18 features $\to$ 8 Category Recommendations (Residential, Commercial, Industrial, Green, etc.)


In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)

gpu = tf.config.list_physical_devices("GPU")
if gpu:
    print("✅ GPU detected:", gpu)
else:
    print("ℹ️ Running on CPU")


In [ ]:
%pip -q install datasets scikit-learn matplotlib joblib pillow


In [ ]:
from datasets import load_dataset
import numpy as np

print("Loading EuroSAT dataset from Hugging Face...")
dataset = load_dataset("tanganke/eurosat")

print("Dataset structure:")
print(dataset)
class_names = dataset["train"].features["label"].names
print("\nEuroSAT Classes (10):")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
for i in range(8):
    sample = dataset["train"][i]
    plt.subplot(2, 4, i + 1)
    plt.imshow(sample["image"])
    plt.title(class_names[sample["label"]])
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split

train_data = dataset["train"]
indices = np.arange(len(train_data))

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=[train_data[int(i)]["label"] for i in indices]
)

print(f"Train samples: {len(train_indices)}")
print(f"Validation samples: {len(val_indices)}")
test_data = dataset["test"] if "test" in dataset else None
if test_data is not None:
    print(f"Test samples: {len(test_data)}")


In [ ]:
def prepare_dataset(hf_dataset, shuffle=False, batch_size=32):
    images = np.array([
        np.array(image.convert("RGB"))
        for image in hf_dataset["image"]
    ], dtype=np.uint8)

    labels = np.array(hf_dataset["label"], dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((images, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=1024, seed=42)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_sub = dataset["train"].select(train_indices.tolist())
val_sub = dataset["train"].select(val_indices.tolist())

train_ds = prepare_dataset(train_sub, shuffle=True)
val_ds = prepare_dataset(val_sub, shuffle=False)
test_ds = prepare_dataset(dataset["test"] if "test" in dataset else val_sub, shuffle=False)

for img_batch, lbl_batch in train_ds.take(1):
    print("Batch Image Shape:", img_batch.shape)
    print("Batch Label Shape:", lbl_batch.shape)


In [ ]:
# ============================================================
# DUAL WATER RECOGNITION DATA GENERATOR
# Real-world satellite scenes often feature both land and water
# (e.g., riverbanks, coastlines, ports). We maintain synthesis support.
# ============================================================

def create_mixed_land_water_sample(land_img, water_img, boundary="horizontal", water_ratio=0.4):
    land_arr = np.array(land_img.convert("RGB"))
    water_arr = np.array(water_img.convert("RGB"))
    h, w, _ = land_arr.shape
    mixed = land_arr.copy()

    if boundary == "horizontal":
        split = int(h * (1.0 - water_ratio))
        mixed[split:, :] = water_arr[split:, :]
    elif boundary == "vertical":
        split = int(w * (1.0 - water_ratio))
        mixed[:, split:] = water_arr[:, split:]
    return mixed

print("✅ Dual Land-Water generator ready")


## Model 1 — PAPNet Architecture
Here we define the **PatternAffinityModule** and **PAPResidualBlock**.

The PatternAffinityModule calculates spatial and feature correlations between patches:
- Maps feature representations into query and key spaces.
- Computes pairwise attention affinity between all spatial grid locations.
- Diffuses pattern representations through the feature space.
- Stabilized with a learnable $\gamma$ residual multiplier.


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

@keras.utils.register_keras_serializable(package="papnet")
class PatternAffinityModule(layers.Layer):
    def __init__(self, key_dim=32, value_dim=None, **kwargs):
        super().__init__(**kwargs)
        self.key_dim = int(key_dim)
        self.value_dim = int(value_dim) if value_dim is not None else None

    def build(self, input_shape):
        channels = int(input_shape[-1])
        v_dim = self.value_dim if self.value_dim is not None else channels

        self.query_conv = layers.Conv2D(self.key_dim, 1, padding="same", use_bias=False, name="pap_q")
        self.key_conv = layers.Conv2D(self.key_dim, 1, padding="same", use_bias=False, name="pap_k")
        self.value_conv = layers.Conv2D(v_dim, 1, padding="same", use_bias=False, name="pap_v")
        self.proj_conv = layers.Conv2D(channels, 1, padding="same", name="pap_proj")
        self.norm = layers.BatchNormalization(name="pap_bn")
        self.gamma = self.add_weight(name="gamma", shape=(1,), initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, x, training=None):
        B = tf.shape(x)[0]
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]
        N = H * W

        q = self.query_conv(x)
        k = self.key_conv(x)
        v = self.value_conv(x)

        q_flat = tf.reshape(q, [B, N, self.key_dim])
        k_flat = tf.reshape(k, [B, N, self.key_dim])
        v_dim = tf.shape(v)[-1]
        v_flat = tf.reshape(v, [B, N, v_dim])

        scale = tf.math.rsqrt(tf.cast(self.key_dim, tf.float32))
        affinity = tf.matmul(q_flat, k_flat, transpose_b=True) * scale
        affinity = tf.nn.softmax(affinity, axis=-1)

        propagated = tf.matmul(affinity, v_flat)
        propagated = tf.reshape(propagated, [B, H, W, v_dim])
        propagated = self.proj_conv(propagated)
        propagated = self.norm(propagated, training=training)

        return x + self.gamma * propagated

    def get_config(self):
        config = super().get_config()
        config.update({"key_dim": self.key_dim, "value_dim": self.value_dim})
        return config


@keras.utils.register_keras_serializable(package="papnet")
class PAPResidualBlock(layers.Layer):
    def __init__(self, filters, stride=1, use_pap=True, key_dim=32, **kwargs):
        super().__init__(**kwargs)
        self.filters = int(filters)
        self.stride = int(stride)
        self.use_pap = bool(use_pap)
        self.key_dim = int(key_dim)

    def build(self, input_shape):
        in_channels = int(input_shape[-1])
        self.conv1 = layers.Conv2D(self.filters, 3, strides=self.stride, padding="same", use_bias=False)
        self.bn1 = layers.BatchNormalization()
        self.act1 = layers.Activation("swish")
        self.conv2 = layers.Conv2D(self.filters, 3, strides=1, padding="same", use_bias=False)
        self.bn2 = layers.BatchNormalization()

        self.pap = PatternAffinityModule(key_dim=self.key_dim) if self.use_pap else None

        if self.stride != 1 or in_channels != self.filters:
            self.shortcut = keras.Sequential([
                layers.Conv2D(self.filters, 1, strides=self.stride, padding="same", use_bias=False),
                layers.BatchNormalization()
            ])
        else:
            self.shortcut = layers.Lambda(lambda x: x)

        self.act_out = layers.Activation("swish")
        super().build(input_shape)

    def call(self, x, training=None):
        res = self.shortcut(x, training=training)
        y = self.act1(self.bn1(self.conv1(x), training=training))
        y = self.bn2(self.conv2(y), training=training)
        if self.pap is not None:
            y = self.pap(y, training=training)
        return self.act_out(y + res)

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "stride": self.stride,
            "use_pap": self.use_pap,
            "key_dim": self.key_dim
        })
        return config

print("✅ PAPNet layers defined successfully!")


In [ ]:
def build_papnet(input_shape=(64, 64, 3), num_classes=10):
    inputs = keras.Input(shape=input_shape, name="satellite_image")

    # Data Augmentation (active during training only)
    x = layers.RandomFlip("horizontal_and_vertical", name="aug_flip")(inputs)
    x = layers.RandomRotation(0.2, name="aug_rot")(x)
    x = layers.RandomZoom(0.2, name="aug_zoom")(x)

    # Rescale [0, 255] -> [0, 1]
    x = layers.Rescaling(1.0 / 255.0, name="rescaling")(x)

    # Stem
    x = layers.Conv2D(32, 3, padding="same", use_bias=False, name="stem_conv")(x)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("swish", name="stem_act")(x)

    # Stage 1: 64x64 -> 32x32
    x = PAPResidualBlock(filters=64, stride=2, use_pap=False, name="stage1_b1")(x)
    x = PAPResidualBlock(filters=64, stride=1, use_pap=True, key_dim=16, name="stage1_b2")(x)

    # Stage 2: 32x32 -> 16x16
    x = PAPResidualBlock(filters=128, stride=2, use_pap=False, name="stage2_b1")(x)
    x = PAPResidualBlock(filters=128, stride=1, use_pap=True, key_dim=32, name="stage2_b2")(x)

    # Stage 3: 16x16 -> 8x8
    x = PAPResidualBlock(filters=256, stride=2, use_pap=False, name="stage3_b1")(x)
    x = PAPResidualBlock(filters=256, stride=1, use_pap=True, key_dim=64, name="stage3_b2")(x)

    # Stage 4: Top Pattern-Affinitive Propagation Refinement (8x8)
    x = PatternAffinityModule(key_dim=64, name="pap_top_refinement")(x)

    # Global Pooling Fusion
    gap = layers.GlobalAveragePooling2D(name="gap")(x)
    gmp = layers.GlobalMaxPooling2D(name="gmp")(x)
    feat = layers.Concatenate(name="pool_cat")([gap, gmp])

    feat = layers.Dense(128, activation="swish", name="head_dense")(feat)
    feat = layers.BatchNormalization(name="head_bn")(feat)
    feat = layers.Dropout(0.3, name="head_dropout")(feat)

    outputs = layers.Dense(num_classes, activation="softmax", name="land_class_probabilities")(feat)

    return keras.Model(inputs=inputs, outputs=outputs, name="PAPNet_EuroSAT")

model = build_papnet()
model.summary()


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("Starting PAPNet Phase 1 Training...")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks
)


In [ ]:
# Fine-Tuning with reduced learning rate for sharp pattern affinity convergence
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
    ]
)
print("✅ PAPNet Phase 2 Fine-Tuning Complete")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_loss, test_acc = model.evaluate(test_ds)
print(f"\n✅ PAPNet Test Accuracy: {test_acc * 100:.2f}% | Test Loss: {test_loss:.4f}")

# Extract true & predicted labels
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    y_pred_probs.append(probs)
    y_true.append(labels.numpy())

y_true = np.concatenate(y_true)
y_pred_probs = np.concatenate(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[c[:10] for c in class_names])
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
plt.title("PAPNet Model 1 Confusion Matrix")
plt.savefig("model1_papnet_confusion_matrix.png")
plt.show()


In [ ]:
import os
os.makedirs("../models", exist_ok=True)

model.save("./model1_papnet.keras")
model.save("../models/model1_papnet.keras")

print("✅ Model 1 (PAPNet) saved to ./model1_papnet.keras and ../models/model1_papnet.keras")


In [ ]:
# ============================================================
# EXTRACT 8 VISUAL FEATURES (MEAN RGB, BRIGHTNESS, INDICES)
# ============================================================

def extract_image_features(images):
    images_norm = images.astype("float32") / 255.0
    features = []

    for img in images_norm:
        r = img[:, :, 0]
        g = img[:, :, 1]
        b = img[:, :, 2]

        mean_r = np.mean(r)
        mean_g = np.mean(g)
        mean_b = np.mean(b)
        brightness = np.mean(img)

        vegetation = np.mean((g > r * 1.05) & (g > b * 1.05))
        water = np.mean((b > r * 1.1) & (b > g * 0.95))
        built_up = np.mean((brightness > 0.45) & (np.abs(r - g) < 0.1) & (np.abs(g - b) < 0.1))
        road = np.mean((np.abs(r - g) < 0.05) & (np.abs(g - b) < 0.05) & (brightness > 0.25) & (brightness < 0.6))

        features.append([
            mean_r, mean_g, mean_b, brightness,
            vegetation, water, built_up, road
        ])

    return np.array(features, dtype=np.float32)

def extract_combined_features(tf_dataset, papnet_model):
    all_features = []
    all_labels = []

    for images, labels in tf_dataset:
        vis_feat = extract_image_features(images.numpy())
        probs = papnet_model.predict(images, verbose=0)
        combined = np.hstack([vis_feat, probs])  # 8 + 10 = 18 features
        all_features.append(combined)
        all_labels.append(labels.numpy())

    return np.concatenate(all_features), np.concatenate(all_labels)

print("Extracting 18 features for training Model 2 and Model 3...")
X_base, y_base = extract_combined_features(train_ds, model)
print("Base Features shape (18-D):", X_base.shape)


In [ ]:
# ============================================================
# MODEL 2 — SITE SUITABILITY PREDICTOR
# ============================================================
from sklearn.preprocessing import StandardScaler
import joblib

def create_suitability_score(features, facility="bank"):
    brightness = features[:, 3]
    vegetation = features[:, 4]
    water = features[:, 5]
    built_up = features[:, 6]
    road = features[:, 7]
    probs = features[:, 8:18]

    forest = probs[:, 1]
    highway = probs[:, 3]
    industrial = probs[:, 4]
    residential = probs[:, 7]
    river = probs[:, 8]
    sea_lake = probs[:, 9]

    if facility == "bank":
        score = (
            residential * 35.0 +
            highway * 25.0 +
            built_up * 20.0 +
            industrial * 10.0 -
            (river + sea_lake + water) * 40.0 -
            forest * 20.0
        )
    else:
        score = (built_up * 30.0 + highway * 20.0 + residential * 20.0 - (water + river) * 30.0)

    return np.clip(score, 0.0, 100.0)

scores = create_suitability_score(X_base, facility="bank")
y2 = (scores >= np.percentile(scores, 50)).astype(np.float32)

X2_train, X2_val, y2_train, y2_val = train_test_split(X_base, y2, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X2_train_scaled = scaler2.fit_transform(X2_train)
X2_val_scaled = scaler2.transform(X2_val)

model2 = keras.Sequential([
    layers.Input(shape=(18,)),
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
], name="Model2_Suitability")

model2.compile(optimizer=keras.optimizers.Adam(0.001), loss="binary_crossentropy", metrics=["accuracy"])
model2.fit(X2_train_scaled, y2_train, validation_data=(X2_val_scaled, y2_val), epochs=15, batch_size=64, verbose=1)

model2.save("./model2_suitability.keras")
model2.save("../models/model2_suitability.keras")
joblib.dump(scaler2, "./model2_scaler.pkl")
joblib.dump(scaler2, "../models/model2_scaler.pkl")
print("✅ Model 2 saved successfully!")


In [ ]:
# ============================================================
# MODEL 3 — 8-CLASS SITE RECOMMENDER
# ============================================================

def create_model3_scores(features):
    brightness = features[:, 3]
    vegetation = features[:, 4]
    water = features[:, 5]
    built_up = features[:, 6]
    road = features[:, 7]

    probs = features[:, 8:18]
    annual_crop = probs[:, 0]
    forest = probs[:, 1]
    shrubland = probs[:, 2]
    highway = probs[:, 3]
    industrial = probs[:, 4]
    pasture = probs[:, 5]
    perm_crop = probs[:, 6]
    residential = probs[:, 7]
    river = probs[:, 8]
    sea_lake = probs[:, 9]

    scores = np.zeros((features.shape[0], 8), dtype=np.float32)
    scores[:, 0] = residential * 0.45 + built_up * 0.25 + highway * 0.15 - (river + sea_lake) * 0.35
    scores[:, 1] = built_up * 0.35 + highway * 0.30 + residential * 0.20 - forest * 0.25
    scores[:, 2] = industrial * 0.50 + highway * 0.30 + built_up * 0.20 - residential * 0.30
    scores[:, 3] = residential * 0.40 + vegetation * 0.25 + built_up * 0.20 - industrial * 0.30
    scores[:, 4] = residential * 0.35 + highway * 0.25 + built_up * 0.25 - industrial * 0.35
    scores[:, 5] = annual_crop * 0.40 + perm_crop * 0.30 + pasture * 0.20 + vegetation * 0.10 - built_up * 0.30
    scores[:, 6] = highway * 0.50 + road * 0.30 + built_up * 0.20 - (river + sea_lake) * 0.30
    scores[:, 7] = forest * 0.45 + shrubland * 0.25 + vegetation * 0.20 + (river + sea_lake) * 0.10 - built_up * 0.40
    return np.clip(scores, 0.0, 1.0)

y3 = create_model3_scores(X_base)
X3_train, X3_val, y3_train, y3_val = train_test_split(X_base, y3, test_size=0.2, random_state=42)

scaler3 = StandardScaler()
X3_train_scaled = scaler3.fit_transform(X3_train)
X3_val_scaled = scaler3.transform(X3_val)

model3 = keras.Sequential([
    layers.Input(shape=(18,)),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(8, activation="sigmoid")
], name="Model3_Recommender")

model3.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse", metrics=["mae"])
model3.fit(X3_train_scaled, y3_train, validation_data=(X3_val_scaled, y3_val), epochs=20, batch_size=64, verbose=1)

model3.save("./model3_recommender.keras")
model3.save("../models/model3_recommender.keras")
joblib.dump(scaler3, "./model3_scaler.pkl")
joblib.dump(scaler3, "../models/model3_scaler.pkl")
print("✅ Model 3 saved successfully!")


In [ ]:
# ============================================================
# END-TO-END PREDICTION PIPELINE VERIFICATION
# ============================================================

def predict_site_papnet(image, facility="Bank"):
    img_rgb = image.convert("RGB")
    img_arr = np.array(img_rgb)
    img_batch = np.expand_dims(img_arr, axis=0)

    # 1. PAPNet prediction
    probs = model.predict(img_batch, verbose=0)[0]
    top_class_idx = np.argmax(probs)
    confidence = float(probs[top_class_idx])
    land_class = class_names[top_class_idx]

    # 2. Extract 8 visual features
    vis_feat = extract_image_features(img_batch)[0]

    # 3. Model 3 Site Recommendation
    feat_18 = np.concatenate([vis_feat, probs]).reshape(1, -1)
    feat_scaled = scaler3.transform(feat_18)
    rec_scores = model3.predict(feat_scaled, verbose=0)[0]

    RECOMMENDATION_NAMES = [
        "Residential Development",
        "Commercial Development",
        "Industrial Development",
        "Educational Development",
        "Healthcare Development",
        "Agricultural Use",
        "Transportation Infrastructure",
        "Green / Conservation"
    ]
    top_rec_idx = np.argmax(rec_scores)
    top_recommendation = RECOMMENDATION_NAMES[top_rec_idx]

    return {
        "land_class": land_class,
        "confidence": confidence,
        "recommendation": top_recommendation,
        "recommendation_score": float(rec_scores[top_rec_idx]),
        "probabilities": {class_names[i]: float(probs[i]) for i in range(10)}
    }

# Test sample
test_sample = dataset["train"][0]["image"]
res = predict_site_papnet(test_sample)
print("\n" + "=" * 50)
print("SITEWISE AI — PAPNET INFERENCE VERIFICATION")
print("=" * 50)
print(f"Land Class: {res['land_class']} ({res['confidence']*100:.1f}%)")
print(f"Top Recommendation: {res['recommendation']} ({res['recommendation_score']:.3f})")
print("=" * 50)
